In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, sum as spark_sum, count, countDistinct
from pyspark.sql.functions import lower, size, split, expr, explode, year, lit, concat
import seaborn as sns
from src.visualization import category_plot
# !pip install spacy 
import spacy
import subprocess
import sys
import re
from scipy.stats import chisquare

In [0]:
# Read the arxiv_bronze table from Unity Catalog
df_whole = spark.table("scientific_trend_analysis.lds.arxiv_bronze")
display(df_whole.limit(5))

In [0]:
# 1. Data Structure & Types
print("\n1. Data Structure & Column Types")
print("-" * 100)
total_rows = df_whole.count()
columns = df_whole.columns
print(f"Total records: {total_rows:,}")
print(f"Total columns: {len(columns)}\n")
print("Column Names and Data Types:")
df_whole.printSchema()


# 2. Missing Values Distribution
print("\n2. Missing Values Distribution")
print("-" * 100)
missing_stats = df_whole.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in columns
])

missing_df = missing_stats.toPandas().T
missing_df.columns = ['missing_count']
missing_df['missing_percentage[%]'] = (missing_df['missing_count'] / total_rows * 100).round(1)
missing_df = missing_df.sort_values('missing_count', ascending=False)

print(f"Columns with missing values:\n")
print(missing_df[missing_df['missing_count'] > 0])
print(f"\nColumns with no missing values:")
print(missing_df[missing_df['missing_count'] == 0].index.tolist())


# 3. Duplicate Records
print("\n3. Duplicate Records Analysis")
print("-" * 100)

# Check for duplicate IDs
if 'id' in columns:
    id_stats = df_whole.agg(
        count('id').alias('id_total'),
        countDistinct('id').alias('id_unique')
    ).collect()[0]
    
    total_ids = id_stats['id_total']
    unique_ids = id_stats['id_unique']
    duplicate_ids = total_ids - unique_ids
    print(f"Duplicate ids: {duplicate_ids:,}")

# Check for duplicate title+abstract combinations
if 'title' in columns and 'abstract' in columns:
    dup_count = df_whole.groupBy('title', 'abstract').count().filter(col('count') > 1).count()
    print(f"Duplicate title + abstract combinations: {dup_count:,}")

In [0]:
df_sub = df_whole.select('id', 'title', 'abstract', 'categories', 'update_date')
display(df_sub.limit(5))

In [0]:
print("Cleaning Steps")
print("-" * 100)

total_before = df_sub.count()
df_no_na = df_sub.dropna(how='all')
removed_na = total_before - df_no_na.count()
print(f"1. Removed {removed_na:,} records with null values")

# Remove duplicate ids
df_no_dup_id = df_no_na.dropDuplicates(subset=['id'])
removed_ids = df_no_na.count() - df_no_dup_id.count()
print(f"2. Removed {removed_ids:,} records with duplicate ids")

# Remove duplicate (title, abstract) pairs
df_cleaned = df_no_dup_id.dropDuplicates(subset=['title', 'abstract'])
removed_content = df_no_dup_id.count() - df_cleaned.count()
print(f"3. Removed {removed_content:,} records with duplicate (title, abstract) pairs")

# Statistics AFTER deduplication
print("\nAfter Cleaning")
print("-" * 100)
total_after = df_cleaned.count()
total_removed = total_before - total_after
removal_pct = (total_removed / total_before) * 100

print(f"Total records: {total_after:,}")
print(f"Total removed: {total_removed:,} ({removal_pct:.2f}%)")
print(f"Data retained: {100 - removal_pct:.2f}%")

# Verify no duplicates remain
verify_stats = df_cleaned.agg(
    countDistinct('id').alias('unique_id'),
    countDistinct('title', 'abstract').alias('unique_title_abstract')
).collect()[0]

print(f"Unique id count: {verify_stats['unique_id']:,}")
print(f"Unique title + abstract count: {verify_stats['unique_title_abstract']:,}")

In [0]:
df_sub = df_sub.withColumn("title", lower(col("title"))) \
                   .withColumn("abstract", lower(col("abstract"))) \
                   .withColumn("categories", lower(col("categories")))

In [0]:
print(f"Abstract Word Count Analysis")
print("-" * 100)

# Create a new column 'abstract_word_count'
df_sub = df_sub.withColumn("abstract_word_count", size(split(("abstract"), " ")))
# Cast explicitly to int to ensure proper type handling
word_count_pd = df_sub.select(col("abstract_word_count").cast("int")).toPandas()

# Show statistics for abstract_word_count
stats_df = df_sub.select("abstract_word_count").describe().toPandas()
mask = stats_df['summary'].isin(['mean', 'stddev'])
stats_df.loc[mask, 'abstract_word_count'] = stats_df.loc[mask, 'abstract_word_count'].astype(float).round(2).astype(str)
display(stats_df)

# Plot box plot of abstract_word_count
plt.figure(figsize=(8, 3))
plt.boxplot(word_count_pd['abstract_word_count'], vert=False)
plt.xlabel('Abstract Word Count')
plt.title('Box Plot of Abstract Word Count')
plt.show()

# Display rows with both small and large outlier
print("\nOutlier Analysis: Abstract Word Count")
print("-" * 100)
q1 = word_count_pd['abstract_word_count'].quantile(0.25)
q3 = word_count_pd['abstract_word_count'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_df_small = df_sub.filter(df_sub.abstract_word_count < lower_bound)
outlier_df_large = df_sub.filter(df_sub.abstract_word_count > upper_bound)
print(f"Small outliers (documents have less than {lower_bound:.0f} words): {outlier_df_small.count():,}")
print(f"Large outliers (documents have more than {upper_bound:.0f} words): {outlier_df_large.count():,}")

#Display top rows with abstract_word_count < 10, excluding abstracts containing 'withdrawn'
print(f"\nDisplay Rows With Word Count Less Than 10 (After Removing Withdrawn Documents)")
print("-" * 100)
df_sub_nonwithdrawn = df_sub.filter(
    (~lower(df_sub.abstract).rlike('withdrawn|withdrown|witdraw|withdraw|witdrawn|withdrwan|withdrawed')) & 
    (~lower(df_sub.abstract).contains('is taken out'))
).select("abstract", "abstract_word_count", "title").orderBy("abstract_word_count").limit(5)
display(df_sub_nonwithdrawn)

### Remove short abstracts for data quality

In [0]:
# Statistics before removal
print("\nBefore Short Abstracts Removal")
print("-" * 100)
total_before = df_sub.count()
short_abstracts = df_sub.filter(col('abstract_word_count') < 10).count()
short_pct = (short_abstracts / total_before) * 100

print(f"Total records: {total_before:,}")
print(f"Papers with < 10 word abstracts: {short_abstracts:,} ({short_pct:.2f}%)")

# Remove short abstracts
df_cleaned = df_sub.filter(col('abstract_word_count') >= 10)

# Statistics after removal
print("\nAfter Removal")
print("-" * 100)
total_after = df_cleaned.count()
total_removed = total_before - total_after
removal_pct = (total_removed / total_before) * 100

print(f"Total records: {total_after:,}")
print(f"Total removed: {total_removed:,} ({removal_pct:.4f}%)")
print(f"Documents retained: {100 - removal_pct:.4f}%")

# Verify minimum word count
min_word_count = df_cleaned.agg({'abstract_word_count': 'min'}).collect()[0][0]
max_word_count = df_cleaned.agg({'abstract_word_count': 'max'}).collect()[0][0]
avg_word_count = df_cleaned.agg({'abstract_word_count': 'avg'}).collect()[0][0]

print(f"\nAbstract word count statistics after cleaning:")
print("-" * 100)
print(f"Minimum: {min_word_count} words")
print(f"Maximum: {max_word_count} words")
print(f"Average: {avg_word_count:.1f} words")

# Update df_whole with cleaned data
print(f"\nClean dataset ready for analysis: {df_cleaned.count():,} papers")

In [0]:
# Save cleaned subset as a Delta table in the Unity Catalog core layer
df_cleaned.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("scientific_trend_analysis.core.cleaned_arxiv_subset")

### Exploratory Data Analysis

In [0]:
df_cleaned = spark.table("scientific_trend_analysis.core.cleaned_arxiv_subset")
display(df_cleaned.limit(5))

In [0]:
df_eda = df_cleaned
df_eda = df_eda.withColumn('category_list', split(col('categories'), ' '))
df_eda = df_eda.withColumn('category_primary', expr("transform(category_list, x -> split(x, '\\\\.')[0])"))
df_eda = df_eda.withColumn('full_text', concat(col('title'), lit(' '), col('abstract')))

df_eda = df_eda.select('id', 'update_date', 'title', 'abstract', 'full_text',
                       'abstract_word_count', 'category_list', 'category_primary')
display(df_eda.limit(5))

In [0]:
# ----- Full category name with subcategory -----
# Explode and count in Spark
df_cat_counts = df_eda.select(explode("category_list").alias("category_list")) \
                      .groupBy("category_list") \
                      .count() \
                      .orderBy(col("count").desc()) \
                      .limit(10)

cat_counts_pd = df_cat_counts.toPandas()

# Extract year and explode category_list, then count
df_year_cat_counts = df_eda.withColumn('year', year(col('update_date'))) \
                            .select('year', explode('category_list').alias('category')) \
                            .groupBy('year', 'category').count() \
                            .orderBy('year', 'category')

year_cat_pd = df_year_cat_counts.toPandas()

# Pivot to get categories as columns
df_pivot = year_cat_pd.pivot(index='year', columns='category', values='count').fillna(0)

# Get top categories by total count for better visualization
top_categories = year_cat_pd.groupby('category')['count'].sum().nlargest(10).index.tolist()
df_pivot_top = df_pivot[top_categories]

category_plot(cat_counts_pd, 'category_list', df_pivot_top)

# ----- Primary category -----
# Explode and count in Spark
df_cat_primary_counts = df_eda.select(explode("category_primary").alias("category_primary")) \
                      .groupBy("category_primary") \
                      .count() \
                      .orderBy(col("count").desc()) \
                      .limit(10)

cat_primary_counts_pd = df_cat_primary_counts.toPandas()

# Extract year and explode category_primary, then count
df_year_cat_primary_counts = df_eda.withColumn('year', year(col('update_date'))) \
                            .select('year', explode('category_primary').alias('category')) \
                            .groupBy('year', 'category').count() \
                            .orderBy('year', 'category')

year_cat_primary_pd = df_year_cat_primary_counts.toPandas()

# Pivot to get categories as columns
df_pivot_primary = year_cat_primary_pd.pivot(index='year', columns='category', values='count').fillna(0)

# Get top categories by total count for better visualization
top_categories_primary = year_cat_primary_pd.groupby('category')['count'].sum().nlargest(10).index.tolist()
df_pivot_top_primary = df_pivot_primary[top_categories_primary]

category_plot(cat_primary_counts_pd, 'category_primary', df_pivot_top_primary)

# Statistics
print(f"\nPublication Trends by Category")
print("-" * 100)
print(f"Years covered: {df_pivot.index.min()} - {df_pivot.index.max()}")
print(f"Total papers: {year_cat_pd['count'].sum():,}")
print(f"Peak year: {df_pivot.sum(axis=1).idxmax()} ({df_pivot.sum(axis=1).max():,.0f} papers)")
print(f"\nTop 5 categories (total papers):")
for i, (cat, count) in enumerate(year_cat_pd.groupby('category')['count'].sum().nlargest(5).items(), 1):
    percentage = (count / year_cat_pd['count'].sum()) * 100
    print(f"  {i}. {cat}: {count:,} papers ({percentage:.2f}%)")
print(f"\nTop 5 primary categories (total papers):")
for i, (cat, count) in enumerate(year_cat_primary_pd.groupby('category')['count'].sum().nlargest(5).items(), 1):
    percentage = (count / year_cat_primary_pd['count'].sum()) * 100
    print(f"  {i}. {cat}: {count:,} papers ({percentage:.2f}%)")

In [0]:
# Save df_eda as a Delta table in the Unity Catalog core layer
df_eda.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("scientific_trend_analysis.core.cleaned_for_eda")

## Silver data preparation for model training

In [0]:
df_cleaned = spark.table("scientific_trend_analysis.core.cleaned_arxiv_subset")
display(df_cleaned.limit(5))

In [0]:
fraction = 100000 / df_cleaned.count()
df_sampled_spark = df_cleaned.sample(withReplacement=False, fraction=fraction, seed=2)
df_sampled = df_sampled_spark.toPandas()

df_sampled['full_text'] = df_sampled['title'] + ' ' + df_sampled['abstract']
df_sampled = df_sampled[['update_date', 'full_text', 'categories']]
df_sampled.head()

In [0]:
# Perform Chi-squared test to check if the category distribution is preserved after sampling
# Get original category distribution
orig_cat_counts = df_cleaned.select(explode(split(col('categories'), ' ')).alias('category')) \
    .groupBy('category').count().orderBy('category').toPandas()

# Get sampled category distribution
sampled_cat_counts = df_sampled_spark.select(explode(split(col('categories'), ' ')).alias('category')) \
    .groupBy('category').count().orderBy('category').toPandas()

# Align categories
all_cats = sorted(set(orig_cat_counts['category']).union(set(sampled_cat_counts['category'])))
orig_counts = orig_cat_counts.set_index('category').reindex(all_cats, fill_value=0)['count'].values
samp_counts = sampled_cat_counts.set_index('category').reindex(all_cats, fill_value=0)['count'].values

# Normalize to proportions (expected frequencies)
orig_props = orig_counts / orig_counts.sum()
samp_obs = samp_counts

# Scale expected frequencies to match sample size
expected = orig_props * samp_obs.sum()

# Chi-squared test
chi2_stat, p_value = chisquare(f_obs=samp_obs, f_exp=expected)

print(f"Chi-squared statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")
if p_value > 0.05:
    print("No significant difference in category distribution after sampling (p > 0.05).")
else:
    print("Significant difference detected in category distribution after sampling (p <= 0.05).")

In [0]:
def clean_noise(text):
    """ Removes LaTeX commands, math symbols, and cleans up spacing. """
    
    text = text.lower()
    
    # Remove all non-letter characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Remove LaTeX command names
    text = re.sub(r'\b(alpha|beta|gamma|delta|epsilon|varepsilon|theta|vartheta|iota|lambda|sigma|varsigma|upsilon|omega|phi|varphi|rho|varrho|tau|mu|nu|xi|omicron|pi|varpi|kappa|varkappa|zeta|eta|chi|psi|aleph|mathbb|mathbf|mathcal|mathrm|frac|sqrt|infty|displaystyle|textrm|textbf|textit|operatorname|subset|subseteq)\b', ' ', text)
    
    # Final space cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [0]:
try:
    nlp_en = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp_en = spacy.load("en_core_web_sm")

spacy_stopwords = nlp_en.Defaults.stop_words
custom_stopwords = {'study', 'result', 'field', 'present', 'function', 'group', 'use', 'method', 'paper', 'theory', 'analysis','research', 'approach', 'article', 'find', 'equation', 'case', 'problem', 'discuss', 'general', 'type', 'solution', 'point', 'observe', 'range', 'value', 'report', 'way', 'rule', 'understand', 'total', 'detail', 'low', 'high', 'model', 'system', 'base', 'propose', 'time', 'task', 'provide', 'performance', 'effect', 'mass', 'datum', 'heavy', 'obtain', 'mode', 'optimal', 'demonstrate', 'achieve'}

final_stopwords = spacy_stopwords.union(custom_stopwords)


In [0]:
# Process in batches to avoid memory buildup
processed_text_list = []
lem_word_list = []

batch_size = 500
total_texts = len(df_sampled)

for i in range(0, total_texts, batch_size):
    batch_end = min(i + batch_size, total_texts)
    print(f"Processing batch {i//batch_size + 1}: texts {i} to {batch_end-1}")
    
    for text in df_sampled['full_text'].iloc[i:batch_end]:
        text = clean_noise(text)
        processed_text = nlp_en(text)
        
        lem_word_sublist = []
        for token in processed_text:
            if not token.is_space and not token.is_punct and token.lemma_ not in final_stopwords:
                lem_word_sublist.append(token.lemma_)

        processed_text = " ".join(lem_word_sublist)    
        processed_text_list.append(processed_text)
        lem_word_list.append(lem_word_sublist)

print(f"Processed {len(processed_text_list)} texts")

In [0]:
df_sampled['lem_word_list'] = lem_word_list
df_sampled['processed_text'] = df_sampled['lem_word_list'].map(lambda words: 
                                                                ' '.join(word for word in words if len(word) > 3))
df_sampled = df_sampled.drop(columns=['lem_word_list'])
df_sampled.head()

In [0]:
df_sampled_spark = spark.createDataFrame(df_sampled)
df_sampled_spark.write.format("delta").mode("overwrite").saveAsTable("scientific_trend_analysis.core.arxiv_cleaned_sampled")